# Exercise 5 - Validate the primordial-to-late-time numerical map

Linear matter spectrum from an existing Boltzmann solver (Lecture 9, notes Sections 6 and 8.4-8.6).
CLASS v3.4.0 (executable, commit `64bbab7`) and CAMB (python). Code: `Library/exercise_5.py`.
Inputs and raw outputs of every CLASS run: `results/exercise_5/class_runs/`. Figures and tables: `results/exercise_5/`.

**Configuration.** The lecture file `class_base.ini` is not in this repository. The working copy
`Library/exercise_5_class_base.ini` rebuilds it from the fiducial model of the notes, eqs. (8.1)-(8.2); all other settings are CLASS defaults.

In [ ]:
%matplotlib widget
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("../Library")
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import exercise_5 as ex5

# Output folders: figures/tables, and a separate folder for the CLASS runs
out_dir = "../results/exercise_5"
run_dir = f"{out_dir}/class_runs"
os.makedirs(run_dir, exist_ok=True)

# Fiducial parameters (working copy of the CLASS configuration)
base = ex5.read_ini()
h = float(base["h"])
A_s, n_s, k_pivot = float(base["A_s"]), float(base["n_s"]), float(base["k_pivot"])

# Range of k used in every comparison: kh in [1e-4, 5] h/Mpc (inside P_k_max = 10 h/Mpc)
kh_min, kh_max = 1e-4, 5.0

## (a) Reconstruction of the matter spectrum

Fiducial run, nonlinear corrections off. From the tables ($k_h$ in $h\,$Mpc$^{-1}$, $P_h$ in $(h^{-1}$Mpc$)^3$):
$k = h k_h$, $P = P_h/h^3$, $T_m = \delta_m/\mathcal{R}$ (CLASS column `d_m`) and

$$P_m^\mathrm{rec}(k) = \frac{2\pi^2}{k^3}A_s\left(\frac{k}{k_*}\right)^{n_s-1}T_m^2(k).$$

In [ ]:
# Run CLASS once with the fiducial configuration (rerun=True: always recompute)
root_fid = ex5.run_class(base, "fiducial", run_dir, rerun=True)

# Derived quantities printed by CLASS in the run log
log = lambda pattern: ex5.read_log_value(root_fid, pattern)
YHe = log(r"Y_He = ([0-9.]+)")
mnu_over_omnu = log(r"m_i / omega_i =([0-9.e+]+) eV")
omega_nu = float(base["m_ncdm"]) / mnu_over_omnu
with open(root_fid[:-1] + ".log") as f:
    version = f.read().split("Running CLASS version ")[1].split()[0]

# Record of the run: code, physical parameters, epoch, gauge, matter component, units
record = {
    "code": f"CLASS {version}, commit {ex5.CLASS_COMMIT}",
    "h": base["h"], "omega_b": base["omega_b"], "omega_cdm": base["omega_cdm"],
    "A_s": base["A_s"], "n_s": base["n_s"], "k_pivot [1/Mpc]": base["k_pivot"],
    "tau_reio": base["tau_reio"],
    "neutrinos": f"N_ncdm = 1, m_ncdm = {base['m_ncdm']} eV, T_ncdm = 0.71611 (default), N_ur = {base['N_ur']}",
    "N_eff": log(r"-> N_eff = ([0-9.]+)"),
    "omega_nu": omega_nu,
    "Y_He (BBN)": YHe,
    "T_cmb [K]": "2.7255 (default)",
    "recombination": "HyRec 2020 (default)",
    "reionization": "tanh in z (reio_camb), width 0.5 (default)",
    "output redshift": base["z_pk"],
    "gauge": base["gauge"],
    "matter component": "total matter: cdm + b + ncdm (d_m, pk.dat)",
    "nonlinear": base["non_linear"],
    "units of the tables": "kh [h/Mpc], Ph [(Mpc/h)^3]; T_m dimensionless, normalized to R = 1",
    "sigma8 (total matter)": log(r"sigma8=([0-9.]+) for total matter"),
}
df_record = pd.DataFrame(record.items(), columns=["quantity", "value"])
df_record.to_csv(f"{out_dir}/tab_run_record.csv", index=False)
df_record

In [ ]:
# Read the CLASS tables (same k grid) and convert the units
tk = ex5.read_class_table(root_fid + "tk.dat")
k, P_class = ex5.class_linear_pk(root_fid, h)        # k [1/Mpc], P [Mpc^3]
assert np.allclose(h * tk["k (h/Mpc)"], k, rtol=1e-14)   # same k grid in both tables

# Reconstruction from the total-matter density transfer
T_m = tk["d_m"]
P_rec = ex5.reconstruct_pk(k, T_m, A_s, n_s, k_pivot)
residual = P_rec / P_class - 1.0

print(f"N_k = {k.size},  max |P_rec/P_CLASS - 1| = {np.max(np.abs(residual)):.3e}"
      f"   (lecture's archived run: 1.884e-12)")

# Save the reconstruction table
pd.DataFrame({"k_1/Mpc": k, "T_m": T_m, "P_class_Mpc3": P_class, "P_rec_Mpc3": P_rec,
              "residual": residual}).to_csv(f"{out_dir}/tab_reconstruction.csv", index=False)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))

# Left: the two spectra, in the h-units of the tables
ax[0].loglog(k / h, P_class * h**3, lw=3, alpha=0.4, label="CLASS direct")
ax[0].loglog(k / h, P_rec * h**3, "k--", lw=1, label="reconstructed")
ax[0].set_xlabel(r"$k$ [$h$ Mpc$^{-1}$]")
ax[0].set_ylabel(r"$P_m(k, z=0)$ [$(h^{-1}$Mpc$)^3$]")
ax[0].set_title("Linear total matter, z = 0, synchronous gauge")
ax[0].legend()

# Right: fractional residual
ax[1].semilogx(k / h, residual, ".-")
ax[1].axhline(0, color="gray", lw=0.5)
ax[1].set_xlabel(r"$k$ [$h$ Mpc$^{-1}$]")
ax[1].set_ylabel(r"$P_m^\mathrm{rec}/P_m^\mathrm{CLASS} - 1$")
ax[1].set_title("Reconstruction residual")

fig.tight_layout()
fig.savefig(f"{out_dir}/fig_reconstruction.pdf")

## (b) Identity check versus convergence

The reconstruction uses the same $T_m$ that CLASS used to build its own $P_m$: it only tests that $A_s$, $n_s$, $k_*$, the $h$-units and the
normalization $T_m = \delta_m/\mathcal{R}$ are read correctly. An error in $T_m$ itself (tolerances, sampling, truncations) cancels, so the residual
is at round-off level whatever the precision. Physical (numerical) accuracy needs runs with different numerical controls.

**Test.** A tighter reference run (`ex5.REFERENCE`: hierarchy depths and switches of CLASS's `pk_ref.pre`, massive-neutrino fluid approximation off), then one control at a time set back to its CLASS default and to a looser value, all others at the
reference; plot $P/P_\mathrm{ref} - 1$. Runs with different $k$ grids are compared on the reference grid with a cubic spline in $\ln P$-$\ln k$
(this interpolation is what the $k$-sampling test measures).

In [ ]:
# Table of the numerical controls actually exposed by CLASS v3.4.0 and their values
rows = []
for name, levels in ex5.CONTROLS.items():
    for par in levels["default"]:
        rows.append({"control": name, "CLASS parameter": par,
                     "reference": ex5.REFERENCE.get(par, "(approx. off)"),
                     "default": levels["default"][par], "loose": levels["loose"][par]})
df_controls = pd.DataFrame(rows)
df_controls.to_csv(f"{out_dir}/tab_controls.csv", index=False)
df_controls

In [ ]:
# Reference run and the CLASS-default run (both with the fiducial physics)
ref_params = {**base, **ex5.REFERENCE}
root_ref = ex5.run_class(ref_params, "reference", run_dir)
k_ref, P_ref = ex5.class_linear_pk(root_ref, h)

# Comparison grid: the reference k nodes inside [kh_min, kh_max]
mask = (k_ref / h >= kh_min) & (k_ref / h <= kh_max)
k_cmp, P_ref_cmp = k_ref[mask], P_ref[mask]

def ratio_to_ref(root):
    # P/P_ref - 1 on the comparison grid
    k_run, P_run = ex5.class_linear_pk(root, h)
    return ex5.interp_pk(k_run, P_run, k_cmp) / P_ref_cmp - 1.0

# All controls at the CLASS defaults = the fiducial run of (a)
ratios = {("all controls", "default"): ratio_to_ref(root_fid)}

# Stability of the reference itself: deeper hierarchies, later switches, more momentum bins
root_tight = ex5.run_class({**ref_params, **ex5.REFERENCE_TIGHTER}, "reference_tighter", run_dir)
ratios[("all controls", "tighter reference")] = ratio_to_ref(root_tight)

# One control at a time (about 15 s per run on one CPU; reused if already on disk)
for name, levels in ex5.CONTROLS.items():
    for level in ["default", "loose"]:
        params = {**ref_params, **levels[level]}
        run_name = "var_" + name.replace(" ", "_").replace("-", "_") + "_" + level
        ratios[(name, level)] = ratio_to_ref(ex5.run_class(params, run_name, run_dir))

# Summary table: largest deviation in two k ranges
rows = []
for (name, level), r in ratios.items():
    low = k_cmp / h < 0.1
    rows.append({"control": name, "level": level,
                 "max_dev_k<0.1": np.max(np.abs(r[low])),
                 "max_dev_0.1<k<5": np.max(np.abs(r[~low]))})
df_conv = pd.DataFrame(rows)
df_conv.to_csv(f"{out_dir}/tab_convergence_class.csv", index=False)
df_conv.style.format({"max_dev_k<0.1": "{:.1e}", "max_dev_0.1<k<5": "{:.1e}"})

In [ ]:
# Number of massive-neutrino momentum bins used in the perturbations (from the run logs)
for level, tol in [("reference", ex5.REFERENCE["tol_ncdm_synchronous"]),
                   ("default", 1e-3), ("loose", 1e-2)]:
    root = root_ref if level == "reference" else f"{run_dir}/var_ncdm_momentum_sampling_{level}_"
    n_q = ex5.read_log_value(root, r"\(resp\. (\d+)\) points for purpose of background")
    print(f"tol_ncdm_synchronous = {tol:.0e} ({level}): {n_q:.0f} momentum bins")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 6), sharex=True, sharey=True)

for ax, name in zip(axes.flat, ["all controls"] + list(ex5.CONTROLS)):
    ax.semilogx(k_cmp / h, ratios[(name, "default")], label="CLASS default")
    if (name, "loose") in ratios:
        ax.semilogx(k_cmp / h, ratios[(name, "loose")], "--", label="looser")
    if (name, "tighter reference") in ratios:
        ax.semilogx(k_cmp / h, ratios[(name, "tighter reference")], ":", label="tighter reference")
    ax.axhline(0, color="gray", lw=0.5)
    ax.set_yscale("symlog", linthresh=1e-5)
    ax.set_title(name, fontsize=10)

axes[0, 0].legend(fontsize=8)
for ax in axes[1]:
    ax.set_xlabel(r"$k$ [$h$ Mpc$^{-1}$]")
for ax in axes[:, 0]:
    ax.set_ylabel(r"$P/P_\mathrm{ref} - 1$")
fig.suptitle("CLASS: one numerical control at a time (others at reference), z = 0")
fig.tight_layout()
fig.savefig(f"{out_dir}/fig_convergence_class.pdf")

**Time sampling at z = 0.** The change is exactly zero: the $z=0$ spectrum is read at the last integration time $\tau_0$, so the
time grid of the stored sources (`perturbations_sampling_stepsize`) is never interpolated. It matters for $z>0$ (check below, CLASS defaults otherwise).

In [ ]:
# Same test at z = 1 (all other settings at the CLASS defaults; runs take ~1 s)
P_z1 = {}
for step in [0.02, 0.1, 0.3]:
    root = ex5.run_class({**base, "z_pk": 1, "perturbations_sampling_stepsize": step},
                         f"z1_stepsize_{step}", run_dir)
    k_z1, P_z1[step] = ex5.class_linear_pk(root, h)
for step in [0.1, 0.3]:
    print(f"z = 1, stepsize {step} vs 0.02: max |dP/P| = {np.max(np.abs(P_z1[step] / P_z1[0.02] - 1)):.1e}")

Controls in the list of the exercise that are **not** varied here:
- *Hierarchy closure*: not an input in CLASS; the scalar photon/massless hierarchies always use the outgoing closure (eq. 2.34 of arXiv:1305.3261) and ncdm the Ma & Bertschinger truncation. Only $\ell_\mathrm{max}$ can be changed.
- *Maximum CMB multipole* (`l_max_scalars`): only enters the $C_\ell$, which are not requested (`output = mPk,mTk`).

## (c) CLASS versus CAMB

Harmonized choices (`ex5.camb_params`):

| | CLASS | CAMB |
|---|---|---|
| parameters | eqs. (8.1)-(8.2), $T_\mathrm{cmb}=2.7255$ K | same |
| neutrinos | 1 massive, $N_\mathrm{eff}=3.044$, $m/\omega_\nu = 93.14$ eV | 1 massive, `nnu = 3.044`, mass rescaled so that `omnuh2` $=\omega_\nu^\mathrm{CLASS}$ |
| helium | BBN value of the CLASS run | same $Y_\mathrm{He}$ passed explicitly |
| recombination | `RECFAST` | Recfast (default) |
| reionization | tanh, $\Delta z = 0.5$ (H and He) | tanh, `delta_redshift = helium_delta_redshift = 0.5` |
| gauge / matter | synchronous, total matter `d_m` | synchronous (CDM frame), `delta_tot` |
| spectrum | linear, $k$ [Mpc$^{-1}$], $P$ [Mpc$^3$] | linear, `hubble_units = k_hunit = False` |

Precision: CLASS reference vs CAMB with `AccuracyBoost = 3`, `k_per_logint = 30`, `accurate_massive_neutrinos = True` (and both at defaults).
Physical-model changes (recombination code, neutrino-mass convention) are shown separately from tolerance changes.

In [ ]:
# CLASS runs with RECFAST (the recombination code also used by CAMB)
recfast = {"recombination": "RECFAST"}
root_ref_rf = ex5.run_class({**ref_params, **recfast}, "reference_recfast", run_dir)
root_def_rf = ex5.run_class({**base, **recfast}, "default_recfast", run_dir)

# CLASS spectra on the comparison grid
P_class_ref_rf = ex5.interp_pk(*ex5.class_linear_pk(root_ref_rf, h), k_cmp)
P_class_def_rf = ex5.interp_pk(*ex5.class_linear_pk(root_def_rf, h), k_cmp)

# CAMB: matched parameters, high precision and default precision
camb_high = dict(accuracy_boost=3, k_per_logint=30, accurate_nu=True)
P_camb_high, res_high = ex5.camb_linear_pk(
    ex5.camb_params({**base, **recfast}, omega_nu, YHe, **camb_high), k_cmp)
P_camb_def, res_def = ex5.camb_linear_pk(
    ex5.camb_params({**base, **recfast}, omega_nu, YHe), k_cmp)

# CAMB at high precision with the naive neutrino definition (m_nu = 0.06 eV passed directly)
pars_naive = ex5.camb_params({**base, **recfast}, omega_nu, YHe, match_omega_nu=False, **camb_high)
P_camb_naive, _ = ex5.camb_linear_pk(pars_naive, k_cmp)
print(f"omega_nu: CLASS = {omega_nu:.6e},  CAMB naive m_nu = 0.06 eV -> {pars_naive.omnuh2:.6e}")
print(f"sigma8:   CLASS ref = {ex5.read_log_value(root_ref_rf, r'sigma8=([0-9.]+) for total'):.6f}, "
      f"CAMB high = {res_high.get_sigma8_0():.6f}")

In [ ]:
# Collect the comparisons
comparisons = {
    # code comparison, matched physics
    "CLASS ref / CAMB high":              P_class_ref_rf / P_camb_high - 1,
    "CLASS default / CAMB default":       P_class_def_rf / P_camb_def - 1,
    # physical-model changes (same code, same precision)
    "CLASS: HyRec / RECFAST (ref)":       P_ref_cmp / P_class_ref_rf - 1,
    "CAMB: naive m_nu / matched (high)":  P_camb_naive / P_camb_high - 1,
    # numerical changes (same code, same physics)
    "CLASS: default / ref (RECFAST)":     P_class_def_rf / P_class_ref_rf - 1,
    "CAMB: default / high":               P_camb_def / P_camb_high - 1,
}

low = k_cmp / h < 0.1
df_codes = pd.DataFrame([{"comparison": name, "max_dev_k<0.1": np.max(np.abs(r[low])),
                          "max_dev_0.1<k<5": np.max(np.abs(r[~low]))}
                         for name, r in comparisons.items()])
df_codes.to_csv(f"{out_dir}/tab_class_camb.csv", index=False)
df_codes.style.format({"max_dev_k<0.1": "{:.1e}", "max_dev_0.1<k<5": "{:.1e}"})

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
names = list(comparisons)
titles = ["CLASS vs CAMB (matched)", "Physical modelling", "Numerical precision"]

for i in range(3):
    for name, ls in zip(names[2 * i: 2 * i + 2], ["-", "--"]):
        ax[i].semilogx(k_cmp / h, comparisons[name], ls, label=name)
    ax[i].axhline(0, color="gray", lw=0.5)
    ax[i].set_yscale("symlog", linthresh=1e-5)
    ax[i].set_xlabel(r"$k$ [$h$ Mpc$^{-1}$]")
    ax[i].set_title(titles[i])
    ax[i].legend(fontsize=8)

ax[0].set_ylabel(r"$P_1/P_2 - 1$")
fig.tight_layout()
fig.savefig(f"{out_dir}/fig_class_camb.pdf")